# One year of prepared XYZ waveforms
Generate two independent eccentric-source waveforms over **365.25 days at 5 s cadence** (6,311,520 samples per channel). Geometry and generation-2 TDI operators are prepared once per padded block and reused for both sources. This is the optimized prepared path, with bounded cache memory. The retained real XYZ arrays for both sources require about 303 MB, plus temporary block arrays.

Use the same installed environment as the minimal example. This example uses analytic LISA orbits, CPU float64, fixed eccentricity, 1PN periastron advance, and no Peters–Mathews evolution. It does not establish Sangria convention compatibility or cadence convergence.


In [1]:
from pathlib import Path
from dataclasses import replace
from time import perf_counter
import sys
import numpy as np
import jax

# Allow running from either the repository root or notebooks/.
root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
if (root / "src").is_dir():
    sys.path.insert(0, str(root / "src"))
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")

from egb_jax_eccentric import (
    EccentricBinaryParams, lisa_orbit, precompute_jax_link_geometry,
    eccentric_links_jax, prepare_xyz_from_links,
)


In [2]:
dt = 5.0  # seconds
samples = int(365.25 * 86400 / dt)
block = 65536
halo = 128  # 640 s of padding on each side; discarded after TDI
source = EccentricBinaryParams(
    mean_motion=np.pi * 1e-3, eccentricity=0.3,
    m1_solar=0.6, m2_solar=0.4, distance_m=3.085677581491367e19,
    beta=0.2, lambda_=0.4, psi=0.3, inclination=0.8, phi0=0.2, fdot=0.0,
)
sources = [source, replace(source, eccentricity=0.4)]

def waveform(source, geometry, tdi):
    return tdi(eccentric_links_jax(
        source, geometry, batch_size=1, physics_mode="1pn_periastron",
    ))

# Warm the exact padded shape; the final block uses this same shape.
warm_state = lisa_orbit(np.arange(-halo, block + halo) * dt)
warm_geometry = precompute_jax_link_geometry(warm_state)
warm_tdi = prepare_xyz_from_links(warm_state, generation=2)
_ = waveform(source, warm_geometry, warm_tdi)
del warm_state, warm_geometry, warm_tdi, _


In [3]:
# Each entry holds an independent source's full-year XYZ (not their sum).
xyz_by_source = [
    {channel: np.empty(samples, dtype=np.float64) for channel in "XYZ"}
    for source in sources
]
setup_s = 0.0
waveform_s = np.zeros(len(sources))
run_start = perf_counter()
for start in range(0, samples, block):
    keep = min(block, samples - start)
    # Padding includes physical times before/after the retained segment.
    times = (start + np.arange(-halo, block + halo)) * dt
    tick = perf_counter()
    state = lisa_orbit(times)
    geometry = precompute_jax_link_geometry(state)
    tdi = prepare_xyz_from_links(
        state, generation=2, measurement_order=3, delay_order=3,
    )
    setup_s += perf_counter() - tick

    for index, source in enumerate(sources):
        tick = perf_counter()
        block_xyz = waveform(source, geometry, tdi)
        waveform_s[index] += perf_counter() - tick
        for channel in "XYZ":
            values = block_xyz[channel][halo:halo + keep]
            assert np.isfinite(values).all()
            assert np.all(values.imag == 0), "Cannot discard a nonzero imaginary component"
            xyz_by_source[index][channel][start:start + keep] = values.real
    del state, geometry, tdi, block_xyz

print(f"Samples per channel: {samples:,}; duration: {samples * dt / 86400:.2f} days")
print(f"Shared geometry + TDI preparation: {setup_s:.3f} s")
for index, seconds in enumerate(waveform_s):
    print(f"e={sources[index].eccentricity}: warmed waveform + TDI {seconds:.3f} s")
print(f"Total loop wall time (including setup, checks, copies): {perf_counter() - run_start:.3f} s")
print({channel: values.shape for channel, values in xyz_by_source[0].items()})


Samples per channel: 6,311,520; duration: 365.25 days
Shared geometry + TDI preparation: 12.271 s
e=0.3: warmed waveform + TDI 3.787 s
e=0.4: warmed waveform + TDI 3.870 s
Total loop wall time (including setup, checks, copies): 19.994 s
{'X': (6311520,), 'Y': (6311520,), 'Z': (6311520,)}


Access the first waveform with `xyz_by_source[0]["X"]` (and `"Y"`, `"Z"`); sample `i` is at `i * dt` seconds. Outputs are dimensionless fractional-frequency Michelson channels.

Per-source timings sum synchronized waveform/link and TDI calls over all blocks. They exclude shared preparation, initial JIT warmup, checks, and output copies. These are single-run demonstration timings, not benchmark medians. Each cache is released after both sources; a later full-year call must prepare blocks again unless their caches were retained. No data files are written.

The 128-sample halo is chosen for this orbit, generation, interpolation order, and cadence; reassess it if those change. The 5 s cadence is illustrative, not a convergence guarantee. pyTDI may emit its existing complex-to-real warning for real GW links stored in complex arrays.
